# No.4 DFT模範解答（教員用）

このノートブックは **教員向けの模範解答** です。学生には最初に見せないでください。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import scipy.io
import time


In [ ]:
N = 256
Dx = 0.1

data = scipy.io.loadmat('../No.2_FFT1D/kukei_DW17.mat')
keys = [k for k in data.keys() if not k.startswith('_')]
g = data[keys[0]].flatten().real[:N]

## 解答1: ループによる素直な実装（教育的）

In [ ]:
re = np.zeros(N)
im = np.zeros(N)

t0 = time.time()
for k in range(N):
    for n in range(N):
        angle = 2 * np.pi * k * n / N
        re[k] += g[n] * np.cos(angle)
        im[k] -= g[n] * np.sin(angle)
G_loop = re + 1j * im
print(f'ループ実装: {time.time()-t0:.2f}秒')

G_ref = np.fft.fft(g)
print(f'np.fft との最大誤差（実部）: {np.max(np.abs(re - G_ref.real)):.2e}')

## 解答2: ベクトル化（高速版）

In [ ]:
n = np.arange(N)
k = np.arange(N)
# 位相行列 (N x N)
W = np.exp(-2j * np.pi * np.outer(k, n) / N)
G_vec = W @ g

print(f'ベクトル化 と np.fft との最大誤差: {np.max(np.abs(G_vec - G_ref)):.2e}')

In [ ]:
freq = np.arange(N) * Dx
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
axes[0].plot(freq, G_ref.real, label='np.fft（FFT）', alpha=0.7)
axes[0].plot(freq, re, '--', label='DFT（ループ）', alpha=0.7)
axes[0].set_title('比較: 実部（一致していれば正解）')
axes[0].legend()
axes[0].grid(True)
axes[1].plot(freq, G_ref.imag, label='np.fft（FFT）', alpha=0.7)
axes[1].plot(freq, im, '--', label='DFT（ループ）', alpha=0.7)
axes[1].set_title('比較: 虚部')
axes[1].legend()
axes[1].grid(True)
fig.tight_layout()
plt.show()